# DATA209 — Advanced Exploratory Data Analysis
# Practical P9-10 · Bivariate analysis and hypothesis generation

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 5 · Module 2 · CO2

---

**Objective.** Build and read a correlation matrix, use heatmaps and cross-tabulation, and convert observed relationships into testable hypotheses.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 and P3-4 — the dataset, the split, and the corrected data types.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P3-4: corrected dtypes
dfc = df.copy()
for c in ["OperatingSystems", "Browser", "Region", "TrafficType"]:
    dfc[c] = dfc[c].astype("category")
_order  = ["Jan","Feb","Mar","Apr","May","June","Jul","Aug","Sep","Oct","Nov","Dec"]
_present = [m for m in _order if m in dfc["Month"].unique()]
dfc["Month"] = pd.Categorical(dfc["Month"], categories=_present, ordered=True)
dfc["VisitorType"] = dfc["VisitorType"].astype(str).str.strip()
for c in ["Weekend", "Revenue"]:
    dfc[c] = dfc[c].astype(bool)

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P9-10 — Bivariate analysis and hypothesis generation

### Correlation matrix

Correlation is standardised covariance, bounded in [-1, 1] and unitless. Pearson measures
**linear** association only; Spearman works on ranks and catches any monotonic relationship.
Comparing the two is a cheap non-linearity test.

In [ ]:
# ---- Correlation matrix, two ways --------------------------------------
num = df[numeric_cols + [TARGET]].copy()
num[TARGET] = num[TARGET].astype(int)

pearson  = num.corr(method="pearson")
spearman = num.corr(method="spearman")

target_corr = pd.DataFrame({
    "pearson" : pearson[TARGET].drop(TARGET),
    "spearman": spearman[TARGET].drop(TARGET),
})
target_corr["gap"] = (target_corr["spearman"] - target_corr["pearson"]).abs()
target_corr = target_corr.reindex(target_corr["spearman"].abs()
                                  .sort_values(ascending=False).index)
print("Association with the target")
print(target_corr.round(3).to_string())

print("\nInterpretation")
print("- A large pearson/spearman gap means the relationship is monotonic but not linear.")
print("- PageValues shows the largest gap: strong in rank terms, weaker as a straight line.")

### Heatmaps

In [ ]:
# ---- Heatmap of the full matrix ----------------------------------------
mask = np.triu(np.ones_like(pearson, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))
sns.heatmap(pearson, mask=mask, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
            vmin=-1, vmax=1, square=False, ax=axes[0], cbar_kws={"shrink": .7},
            annot_kws={"size": 7})
axes[0].set_title("Pearson correlation")

sns.heatmap(spearman, mask=mask, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
            vmin=-1, vmax=1, square=False, ax=axes[1], cbar_kws={"shrink": .7},
            annot_kws={"size": 7})
axes[1].set_title("Spearman correlation")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Multicollinearity among the predictors ----------------------------
pred = pearson.drop(index=TARGET, columns=TARGET)
pairs = (pred.where(np.triu(np.ones(pred.shape), k=1).astype(bool))
             .stack().rename("r").reset_index())
pairs.columns = ["var_1", "var_2", "r"]
pairs["abs_r"] = pairs["r"].abs()
print("Most correlated predictor pairs")
print(pairs.sort_values("abs_r", ascending=False).head(8).round(3).to_string(index=False))

# VIF from the inverse correlation matrix — no statsmodels needed
def vif_table(frame):
    c = frame.corr().values
    inv = np.linalg.pinv(c)
    return (pd.Series(np.diag(inv), index=frame.columns, name="VIF")
              .sort_values(ascending=False).to_frame())

vif = vif_table(df[numeric_cols])
vif["verdict"] = pd.cut(vif["VIF"], [0, 5, 10, np.inf],
                        labels=["ok", "investigate", "severe"])
print("\nVariance inflation factor")
print(vif.round(2).to_string())
print("\nVIF > 10 means the column is largely reconstructable from the others.")
print("ExitRates and BounceRates measure nearly the same thing — keep one, or combine them.")

### Cross-tab analysis

For two categorical variables, counts mislead and **percentages inform**. Always normalise along
the axis that answers your question, and always report the group sizes.

In [ ]:
# ---- Cross-tabulation ---------------------------------------------------
def crosstab_report(frame, cat, target=TARGET, min_n=30):
    counts = pd.crosstab(frame[cat], frame[target])
    rate   = pd.crosstab(frame[cat], frame[target], normalize="index") * 100
    out = pd.DataFrame({
        "sessions"      : counts.sum(axis=1),
        "purchases"     : counts.get(True, 0),
        "conversion_%"  : rate.get(True, 0).round(2),
    })
    out["reliable"] = out["sessions"] >= min_n
    return out.sort_values("conversion_%", ascending=False)

for c in ["VisitorType", "Weekend", "Month"]:
    print(f"--- {c} vs {TARGET}")
    print(crosstab_report(dfc, c).to_string(), "\n")

In [ ]:
# ---- Chi-square and effect size ----------------------------------------
from scipy.stats import chi2_contingency

def cramers_v(frame, a, b):
    tab = pd.crosstab(frame[a], frame[b])
    chi2, p, dof, _ = chi2_contingency(tab)
    n = tab.values.sum()
    v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
    return chi2, p, v

rows = []
for c in ["VisitorType", "Weekend", "Month", "Region", "TrafficType", "Browser"]:
    chi2, p, v = cramers_v(dfc, c, TARGET)
    rows.append({"variable": c, "chi2": chi2, "p_value": p, "cramers_v": v,
                 "strength": "negligible" if v < .1 else
                             "weak" if v < .2 else
                             "moderate" if v < .3 else "strong"})
assoc = pd.DataFrame(rows).sort_values("cramers_v", ascending=False)
print(assoc.round(4).to_string(index=False))

print("\nInterpretation")
print("- Chi-square says whether an association exists; Cramer's V says how strong it is.")
print("- With 12,330 rows almost everything is 'significant'. Read the effect size, not the p-value.")

In [ ]:
# ---- Visualise the strongest relationships -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))

ct = pd.crosstab(dfc["VisitorType"], dfc[TARGET], normalize="index") * 100
ct.plot(kind="bar", stacked=True, ax=axes[0],
        color=["#C9D4DB", "#1F6F6B"], legend=False)
axes[0].set_title("Conversion by visitor type (%)"); axes[0].set_xlabel("")

sns.boxplot(data=df, x=TARGET, y="PageValues", ax=axes[1], showfliers=False)
axes[1].set_title("PageValues by outcome")

sns.scatterplot(data=df.sample(3000, random_state=RANDOM_STATE),
                x="BounceRates", y="ExitRates", hue=TARGET,
                alpha=0.4, s=12, ax=axes[2])
axes[2].set_title("BounceRates vs ExitRates (r = %.2f)"
                  % df["BounceRates"].corr(df["ExitRates"]))
plt.tight_layout(); plt.show()

### Hypothesis generation

A hypothesis names a **mechanism**, predicts a **direction**, and states what would **falsify** it.
"X is related to Y" is a restatement of the correlation, not a hypothesis.

In [ ]:
# ---- Hypotheses from the evidence above --------------------------------
hypotheses = [
    dict(id="H1",
         claim="Sessions that reach a page carrying a positive PageValue are far more likely to "
               "convert, because PageValues is assigned to pages on the checkout path.",
         direction="PageValues > 0 raises conversion",
         evidence="Spearman with target is the largest of any variable; medians differ sharply",
         falsified_by="Converting and non-converting sessions show the same PageValues "
                      "distribution once session depth is controlled for",
         confounder="Session depth — deeper sessions reach more pages of every kind"),
    dict(id="H2",
         claim="Returning visitors convert at a higher rate than new visitors because they have "
               "already evaluated the retailer on an earlier visit.",
         direction="Returning_Visitor > New_Visitor",
         evidence="Cross-tab conversion rates differ by visitor type; Cramer's V is non-negligible",
         falsified_by="The difference disappears after conditioning on month and traffic type",
         confounder="Acquisition channel: new visitors may arrive from lower-intent campaigns"),
    dict(id="H3",
         claim="Conversion is seasonal, peaking in the November-December shopping period.",
         direction="Nov and Dec above the annual mean",
         evidence="Monthly conversion series is not flat; volume also rises",
         falsified_by="Monthly rates fall within sampling variation of the overall rate",
         confounder="Promotional calendar is not observed in this dataset"),
]

for h in hypotheses:
    print(f"[{h['id']}] {textwrap.fill(h['claim'], 92, subsequent_indent='     ')}")
    print(f"     direction    : {h['direction']}")
    print(f"     evidence     : {h['evidence']}")
    print(f"     falsified by : {textwrap.fill(h['falsified_by'], 74, subsequent_indent=' '*20)}")
    print(f"     confounder   : {h['confounder']}\n")

### Deliverable — P9-10

A notebook ending in a **numbered list of at least three hypotheses**, each with its direction,
supporting evidence, falsification condition and named confounders.